# OpenSSL: keys and file formats

Run cells from top to bottom. Generated keys are random, so their values differ between runs.

Written answers (**Your answer** cells) are not checked automatically – after you finish a task, verify them against the notebook in `solutions/`.

> **Solution notebook.** Values in the answers come from the run saved in this notebook. Re-running it generates new keys, so the values change; the explanations stay valid.

In [1]:
!openssl version

OpenSSL 3.5.5 27 Jan 2026 (Library: OpenSSL 3.5.5 27 Jan 2026)


## Generate the working files
Run this cell once. It creates RSA and EC keys in PEM and DER. Encryption, signatures and key exchange follow in `05_rsa_from_scratch.ipynb` and `06_ec_from_scratch.ipynb`.

In [2]:
!openssl genpkey -algorithm RSA -outform PEM -pkeyopt rsa_keygen_bits:2048 -pkeyopt rsa_keygen_pubexp:3 -out RSA_2048_priv.pem
!openssl pkey -inform PEM -outform DER -in RSA_2048_priv.pem -out RSA_2048_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in RSA_2048_priv.pem -out RSA_2048_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in RSA_2048_pub.pem -out RSA_2048_pub.der

!openssl genpkey -algorithm EC -outform PEM -pkeyopt ec_paramgen_curve:P-256 -pkeyopt ec_param_enc:named_curve -out EC_256_priv.pem
!openssl pkey -inform PEM -outform DER -in EC_256_priv.pem -out EC_256_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in EC_256_priv.pem -out EC_256_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in EC_256_pub.pem -out EC_256_pub.der

.............++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*............................................................................

.............................................................................................++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*................................

....................................................++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*...............................................................

............................++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
...........................................+++++++++++++++++++++++++++++++++++++++

+++++++++++++++++++++++++*...................................................................................................................................

..........................................................................................++++++++++++++++++++++++++++++

++++++++++++++++++++++++++++++++++*..............................................................................................

....................................................................................................................................

.......................................................................++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


## Tasks
Useful commands:
- `openssl asn1parse -i -inform DER -in FILE` prints a DER ASN.1 structure.
- `openssl pkey -text -noout -in FILE` prints key details. Add `-pubin` for a public key.

### Task 1: Convert PEM to DER
Run the cell below. It decodes `RSA_2048_pub.pem` and verifies that the result is exactly `RSA_2048_pub.der`.

In [3]:
import base64

pem_lines = open("RSA_2048_pub.pem").read().splitlines()
assert pem_lines[0].startswith("-----BEGIN ") and pem_lines[0].endswith("-----")
assert pem_lines[-1].startswith("-----END ") and pem_lines[-1].endswith("-----")
assert pem_lines[0][11:-5] == pem_lines[-1][9:-5]
der_from_pem = base64.b64decode("".join(pem_lines[1:-1]), validate=True)
der_file = open("RSA_2048_pub.der", "rb").read()

print(der_from_pem == der_file)
assert der_from_pem == der_file

True


**a)** What is the role of Base64 in the PEM representation?

**b)** What should be verified in the PEM header/footer before sharing the key?

**Answer**

**a)** Base64 turns the binary DER bytes into printable ASCII, so the key can be sent through a text-only channel (e-mail, copy & paste, configuration files).

**b)** Check that the PEM label says `PUBLIC KEY`, not `PRIVATE KEY`, so you verify that a public key – not a private key – is being shared.

### Task 2: Read a DER RSA public key
Run the cells in order. Start with the ASN.1 structure; the raw bytes come last.

In [4]:
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der

    0:d=0  hl=4 l= 288 cons: SEQUENCE          
    4:d=1  hl=2 l=  13 cons:  SEQUENCE          
    6:d=2  hl=2 l=   9 prim:   OBJECT            :rsaEncryption
   17:d=2  hl=2 l=   0 prim:   NULL              
   19:d=1  hl=4 l= 269 prim:  BIT STRING        


**a)** Find the algorithm OID (`OBJECT`) and the `BIT STRING` in the output above. At which offsets do they start? The offset is the first number on each line.

Write the two offsets into the code cell below (replace `...`) and run it.

In [ ]:
# outer SEQUENCE at offset 0: 30 82 01 20 – 2-byte long-form length, 288 bytes of content
oid_offset        = 6   # rsaEncryption (1.2.840.113549.1.1.1), followed by NULL (no parameters)
bit_string_offset = 19

der = open("RSA_2048_pub.der", "rb").read()
assert der[oid_offset] == 0x06, "no OBJECT IDENTIFIER tag (06) at oid_offset"
assert der[bit_string_offset] == 0x03, "no BIT STRING tag (03) at bit_string_offset"
print("OK")

In [5]:
# parses the content of the BIT STRING; uses bit_string_offset from the cell above
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der -strparse {bit_string_offset}

    0:d=0  hl=4 l= 264 cons: SEQUENCE          
    4:d=1  hl=4 l= 257 prim:  INTEGER           :D2B824FB6BA87929C61F10389E2CF892DA44267B802A09CAD0D2D72CA3442EFD1BE90987EFC5C9DC1B8E430A2D6BF48AA889F512734C0319208685A921B57A84DE4662E91CF296CBBE410FEDEB395D7EA091C07E6890D20CFCAC14540E8701913BA1AA6D328357AEAB49C8875B12B50E3A550E08C5D4D8DFA662A4C0E0F9C9330BC5CE5065E39282624F16EFC5CD06F1A12B3B6BAFF710A4254901EF2D3E31DC613C3DB11F40298373652DBB9BFFA7FA5824CDA566FA7548A843E10592D69F529AF9791B5A6E9CF1ECBB2E727A0885849CB76717B9C7B3BBC58BC8EE720B907E9FA3C11F48B11B3FBC269BA333CFC23A1869630DAF751DC672B819D3056F35D7
  265:d=1  hl=2 l=   1 prim:  INTEGER           :03


**b)** What two values does the nested structure contain, and which ASN.1 type stores them?

**Answer**

**b)** `SEQUENCE { INTEGER n, INTEGER e }` – the RSA public key: the modulus and the public exponent, both stored as `INTEGER`.

In [6]:
der = open("RSA_2048_pub.der", "rb").read()
for offset in range(0, len(der), 16):
    chunk = der[offset:offset + 16]
    print(f"{offset:03}: {chunk.hex(' ')}")

000: 30 82 01 20 30 0d 06 09 2a 86 48 86 f7 0d 01 01
016: 01 05 00 03 82 01 0d 00 30 82 01 08 02 82 01 01
032: 00 d2 b8 24 fb 6b a8 79 29 c6 1f 10 38 9e 2c f8
048: 92 da 44 26 7b 80 2a 09 ca d0 d2 d7 2c a3 44 2e
064: fd 1b e9 09 87 ef c5 c9 dc 1b 8e 43 0a 2d 6b f4
080: 8a a8 89 f5 12 73 4c 03 19 20 86 85 a9 21 b5 7a
096: 84 de 46 62 e9 1c f2 96 cb be 41 0f ed eb 39 5d
112: 7e a0 91 c0 7e 68 90 d2 0c fc ac 14 54 0e 87 01
128: 91 3b a1 aa 6d 32 83 57 ae ab 49 c8 87 5b 12 b5
144: 0e 3a 55 0e 08 c5 d4 d8 df a6 62 a4 c0 e0 f9 c9
160: 33 0b c5 ce 50 65 e3 92 82 62 4f 16 ef c5 cd 06
176: f1 a1 2b 3b 6b af f7 10 a4 25 49 01 ef 2d 3e 31
192: dc 61 3c 3d b1 1f 40 29 83 73 65 2d bb 9b ff a7
208: fa 58 24 cd a5 66 fa 75 48 a8 43 e1 05 92 d6 9f
224: 52 9a f9 79 1b 5a 6e 9c f1 ec bb 2e 72 7a 08 85
240: 84 9c b7 67 17 b9 c7 b3 bb c5 8b c8 ee 72 0b 90
256: 7e 9f a3 c1 1f 48 b1 1b 3f bc 26 9b a3 33 cf c2
272: 3a 18 69 63 0d af 75 1d c6 72 b8 19 d3 05 6f 35
288: d7 02 01 03


**c)** Locate the modulus and the public exponent in the hex dump above. Each of them is encoded as tag, length, value.

Write your findings into the code cell below (replace `...`) and run it.

In [ ]:
modulus_offset = 28          # tag 02 = INTEGER
modulus_length = 257         # length bytes 82 01 01; the value is the sign byte 00 followed by 256 bytes of n
exponent_tlv   = "02 01 03"  # tag INTEGER, length 1, value 3 (offset 289)

assert der[modulus_offset] == 0x02, "no INTEGER tag (02) at modulus_offset"
assert modulus_length == int.from_bytes(der[modulus_offset + 2:modulus_offset + 4], "big"), "modulus length is wrong"
assert bytes.fromhex(exponent_tlv) == der[-3:], "exponent bytes are wrong"
print("OK")

### Task 3: Inspect RSA and EC key details
Run each cell and answer the questions below it.

In [7]:
!openssl pkey -inform PEM -text -noout -in RSA_2048_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in RSA_2048_pub.pem

Private-Key: (2048 bit, 2 primes)
modulus:
    00:d2:b8:24:fb:6b:a8:79:29:c6:1f:10:38:9e:2c:
    f8:92:da:44:26:7b:80:2a:09:ca:d0:d2:d7:2c:a3:
    44:2e:fd:1b:e9:09:87:ef:c5:c9:dc:1b:8e:43:0a:
    2d:6b:f4:8a:a8:89:f5:12:73:4c:03:19:20:86:85:
    a9:21:b5:7a:84:de:46:62:e9:1c:f2:96:cb:be:41:
    0f:ed:eb:39:5d:7e:a0:91:c0:7e:68:90:d2:0c:fc:
    ac:14:54:0e:87:01:91:3b:a1:aa:6d:32:83:57:ae:
    ab:49:c8:87:5b:12:b5:0e:3a:55:0e:08:c5:d4:d8:
    df:a6:62:a4:c0:e0:f9:c9:33:0b:c5:ce:50:65:e3:
    92:82:62:4f:16:ef:c5:cd:06:f1:a1:2b:3b:6b:af:
    f7:10:a4:25:49:01:ef:2d:3e:31:dc:61:3c:3d:b1:
    1f:40:29:83:73:65:2d:bb:9b:ff:a7:fa:58:24:cd:
    a5:66:fa:75:48:a8:43:e1:05:92:d6:9f:52:9a:f9:
    79:1b:5a:6e:9c:f1:ec:bb:2e:72:7a:08:85:84:9c:
    b7:67:17:b9:c7:b3:bb:c5:8b:c8:ee:72:0b:90:7e:
    9f:a3:c1:1f:48:b1:1b:3f:bc:26:9b:a3:33:cf:c2:
    3a:18:69:63:0d:af:75:1d:c6:72:b8:19:d3:05:6f:
    35:d7
publicExponent: 3 (0x3)
privateExponent:
    00:8c:7a:c3:52:47:c5:a6:1b:d9:6a:0a:d0:69:73:
    50

Public-Key: (2048 bit)
Modulus:
    00:d2:b8:24:fb:6b:a8:79:29:c6:1f:10:38:9e:2c:
    f8:92:da:44:26:7b:80:2a:09:ca:d0:d2:d7:2c:a3:
    44:2e:fd:1b:e9:09:87:ef:c5:c9:dc:1b:8e:43:0a:
    2d:6b:f4:8a:a8:89:f5:12:73:4c:03:19:20:86:85:
    a9:21:b5:7a:84:de:46:62:e9:1c:f2:96:cb:be:41:
    0f:ed:eb:39:5d:7e:a0:91:c0:7e:68:90:d2:0c:fc:
    ac:14:54:0e:87:01:91:3b:a1:aa:6d:32:83:57:ae:
    ab:49:c8:87:5b:12:b5:0e:3a:55:0e:08:c5:d4:d8:
    df:a6:62:a4:c0:e0:f9:c9:33:0b:c5:ce:50:65:e3:
    92:82:62:4f:16:ef:c5:cd:06:f1:a1:2b:3b:6b:af:
    f7:10:a4:25:49:01:ef:2d:3e:31:dc:61:3c:3d:b1:
    1f:40:29:83:73:65:2d:bb:9b:ff:a7:fa:58:24:cd:
    a5:66:fa:75:48:a8:43:e1:05:92:d6:9f:52:9a:f9:
    79:1b:5a:6e:9c:f1:ec:bb:2e:72:7a:08:85:84:9c:
    b7:67:17:b9:c7:b3:bb:c5:8b:c8:ee:72:0b:90:7e:
    9f:a3:c1:1f:48:b1:1b:3f:bc:26:9b:a3:33:cf:c2:
    3a:18:69:63:0d:af:75:1d:c6:72:b8:19:d3:05:6f:
    35:d7
Exponent: 3 (0x3)


**a)** Record the first and last 8 hexadecimal digits of RSA `n`, `e`, `d`, `p`, and `q` (notebook 04 decodes the full values automatically). Explain the leading `00:` byte – compare with the INTEGER encoding on the slides (128 → `00 80`).

Write this into the **Your answer** cell, then fill in the two numbers in the code cell below it.

**Answer**

**a)** n = `00d2b824…056f35d7`, e = `3 (0x3)`, d = `008c7ac3…08ff5deb`, p = `00ec0098…4f0c25d1`, q = `00e4931b…a8e40327`

Why the printed `n` starts with `00:`: a DER `INTEGER` is signed. The top bit of n is 1, so a `00` byte is added to keep the number positive (257 bytes printed, 2048 bits of value).

In [ ]:
n_bits          = 2048
n_printed_bytes = 257  # 256 bytes of n + the leading 00

from cryptography.hazmat.primitives.serialization import load_pem_private_key, Encoding, PublicFormat
rsa = load_pem_private_key(open("RSA_2048_priv.pem", "rb").read(), None)
n = rsa.public_key().public_numbers().n
assert n_bits == n.bit_length(), "bit length of n is wrong"
assert n_printed_bytes == n.bit_length() // 8 + 1, "number of printed bytes is wrong"
print("OK")

In [8]:
!openssl pkey -inform PEM -text -noout -in EC_256_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in EC_256_pub.pem

Private-Key: (256 bit)
priv:
    a5:81:b7:c9:6d:8c:89:d4:92:77:67:63:6f:69:f7:
    21:6e:3e:07:f8:ac:33:05:7d:cd:43:6c:67:95:9e:
    89:a5
pub:
    04:c9:e0:aa:1f:d4:e9:52:21:b6:dc:cf:da:60:c7:
    4d:19:17:90:82:91:8b:be:7a:4f:08:53:c0:53:be:
    60:8f:df:52:4d:de:d5:84:d0:a8:75:08:7b:1f:ae:
    ac:02:10:f0:24:76:31:0b:47:cf:ef:8c:8c:17:ee:
    6e:71:1b:2c:cb
ASN1 OID: prime256v1
NIST CURVE: P-256


Public-Key: (256 bit)
pub:
    04:c9:e0:aa:1f:d4:e9:52:21:b6:dc:cf:da:60:c7:
    4d:19:17:90:82:91:8b:be:7a:4f:08:53:c0:53:be:
    60:8f:df:52:4d:de:d5:84:d0:a8:75:08:7b:1f:ae:
    ac:02:10:f0:24:76:31:0b:47:cf:ef:8c:8c:17:ee:
    6e:71:1b:2c:cb
ASN1 OID: prime256v1
NIST CURVE: P-256


**b)** Record the EC private key, public point, and curve. The public point is stored as `04 ‖ x ‖ y` (`04` = uncompressed point). Why do x and y have this size?

Write this into the **Your answer** cell, then fill in the two numbers in the code cell below it.

**Answer**

**b)** priv = `a581b7c9…959e89a5` (32 bytes), pub = `04c9e0aa…711b2ccb` (65 bytes), curve = `prime256v1` = NIST P-256

Size of x and y: 32 bytes each – they are numbers modulo the 256-bit prime p of P-256, so the point has 1 (`04`) + 32 + 32 = 65 bytes.

In [ ]:
point_bytes      = 65  # 04 ‖ x ‖ y
coordinate_bytes = 32

ec = load_pem_private_key(open("EC_256_priv.pem", "rb").read(), None)
point = ec.public_key().public_bytes(Encoding.X962, PublicFormat.UncompressedPoint)
assert point_bytes == len(point), "length of the public point is wrong"
assert coordinate_bytes == ec.curve.key_size // 8, "length of x and y is wrong"
print("OK")

In [9]:
!openssl asn1parse -i -inform DER -in RSA_2048_priv.der

    0:d=0  hl=4 l=1188 cons: SEQUENCE          
    4:d=1  hl=2 l=   1 prim:  INTEGER           :00
    7:d=1  hl=4 l= 257 prim:  INTEGER           :D2B824FB6BA87929C61F10389E2CF892DA44267B802A09CAD0D2D72CA3442EFD1BE90987EFC5C9DC1B8E430A2D6BF48AA889F512734C0319208685A921B57A84DE4662E91CF296CBBE410FEDEB395D7EA091C07E6890D20CFCAC14540E8701913BA1AA6D328357AEAB49C8875B12B50E3A550E08C5D4D8DFA662A4C0E0F9C9330BC5CE5065E39282624F16EFC5CD06F1A12B3B6BAFF710A4254901EF2D3E31DC613C3DB11F40298373652DBB9BFFA7FA5824CDA566FA7548A843E10592D69F529AF9791B5A6E9CF1ECBB2E727A0885849CB76717B9C7B3BBC58BC8EE720B907E9FA3C11F48B11B3FBC269BA333CFC23A1869630DAF751DC672B819D3056F35D7
  268:d=1  hl=2 l=   1 prim:  INTEGER           :03
  271:d=1  hl=4 l= 257 prim:  INTEGER           :8C7AC35247C5A61BD96A0AD069735061E6D819A7AAC6B131E08C8F731782C9FE129B5BAFF52E8692BD098206C8F2A3071B06A361A232ACBB6B04591B6BCE51ADE98441F0BDF70F327ED60A9E9CD0E8FF15B6805445B5E15DFDC80D8D5F04ABB627C11C48CC578FC9C78685AF920C78B426E35EB083E33

In [10]:
!openssl asn1parse -i -inform DER -in EC_256_priv.der

    0:d=0  hl=2 l= 119 cons: SEQUENCE          
    2:d=1  hl=2 l=   1 prim:  INTEGER           :01
    5:d=1  hl=2 l=  32 prim:  OCTET STRING      [HEX DUMP]:A581B7C96D8C89D4927767636F69F7216E3E07F8AC33057DCD436C67959E89A5
   39:d=1  hl=2 l=  10 cons:  cont [ 0 ]        
   41:d=2  hl=2 l=   8 prim:   OBJECT            :prime256v1
   51:d=1  hl=2 l=  68 cons:  cont [ 1 ]        
   53:d=2  hl=2 l=  66 prim:   BIT STRING        


**c)** Which values do the `INTEGER`s of `RSA_2048_priv.der` hold? Then find the RSA private exponent `d` (`INTEGER`), the EC private key (`OCTET STRING`), the curve identifier (`OBJECT`) and the public point (`BIT STRING`) in the two outputs above.

List the `INTEGER`s in the **Your answer** cell, then write the four offsets into the code cell below it.

**Answer**

**c)** RSA `INTEGER`s in order: version (offset 4), n (offset 7), e, d, p, q, exponent1, exponent2, coefficient – all inside the `SEQUENCE` at offset 0. The EC key is `SEQUENCE { INTEGER 1, OCTET STRING (private key), [0] OBJECT prime256v1, [1] BIT STRING (public point) }`.

In [ ]:
rsa_d_offset    = 271
ec_priv_offset  = 5
ec_curve_offset = 41  # inside the [0] wrapper at offset 39
ec_point_offset = 53  # inside the [1] wrapper at offset 51

rsa_der = open("RSA_2048_priv.der", "rb").read()
ec_der = open("EC_256_priv.der", "rb").read()
d = rsa.private_numbers().d
d = d.to_bytes((d.bit_length() + 7) // 8, "big")
priv = ec.private_numbers().private_value.to_bytes(32, "big")
assert rsa_der[rsa_d_offset] == 0x02 and d in rsa_der[rsa_d_offset:rsa_d_offset + 5 + len(d)], "d does not start at rsa_d_offset"
assert ec_der[ec_priv_offset] == 0x04 and priv in ec_der[ec_priv_offset:ec_priv_offset + 2 + len(priv)], "the EC private key does not start at ec_priv_offset"
assert ec_der[ec_curve_offset] == 0x06, "no OBJECT IDENTIFIER tag (06) at ec_curve_offset"
assert ec_der[ec_point_offset] == 0x03, "no BIT STRING tag (03) at ec_point_offset"
print("OK")